# Cat vs Dog Image Classification using SVM

**Task:** Implement a Support Vector Machine (SVM) to classify images of cats and dogs from the Kaggle 'Dogs vs. Cats' dataset.

This notebook covers:
1. Load images
2. Preprocess (resize, grayscale)
3. Extract features (HOG - Histogram of Oriented Gradients)
4. Split into train/test sets
5. Train an SVM classifier
6. Evaluate performance (accuracy, confusion matrix, classification report)
7. Predict on a new image


## 1. Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
IMG_SIZE = 64  # resize all images to 64x64 for speed


## 2. Load the Dataset

**To use the real Kaggle 'Dogs vs. Cats' dataset:**
1. Download it from https://www.kaggle.com/c/dogs-vs-cats/data (or `kaggle competitions download -c dogs-vs-cats`)
2. Extract it so you have a folder structure like:
```
train/
  cat.0.jpg, cat.1.jpg, ...
  dog.0.jpg, dog.1.jpg, ...
```
3. Replace the next cell's loading code with:
```python
import cv2

def load_images_from_folder(folder, label, limit=1000):
    images, labels = [], []
    files = [f for f in os.listdir(folder) if f.startswith(label)][:limit]
    for fname in files:
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        images.append(img)
        labels.append(0 if label == 'cat' else 1)
    return images, labels

cat_imgs, cat_labels = load_images_from_folder('train', 'cat')
dog_imgs, dog_labels = load_images_from_folder('train', 'dog')
images = cat_imgs + dog_imgs
labels = cat_labels + dog_labels
```

For now, we'll generate **synthetic grayscale images** with distinct textures/shapes to represent two classes ('cat'=0, 'dog'=1), so the full pipeline runs end-to-end without needing to download anything. The code structure below is identical to what you'd use for real images — you're just swapping the data source.

In [ ]:
# --- Synthetic image generator (replace with real Kaggle images using the code above) ---

def make_synthetic_image(label, size=IMG_SIZE):
    """Generates a simple synthetic grayscale image with class-dependent texture/shape."""
    img = np.random.normal(120, 15, (size, size))

    yy, xx = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2

    if label == 0:  # 'cat'-like: rounder blob + pointed ears (triangular corners)
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        img += np.where(dist < size * 0.32, 60, 0)
        img[0:size // 4, 0:size // 4] += 40  # ear
        img[0:size // 4, -size // 4:] += 40  # ear
    else:  # 'dog'-like: elongated blob (wider, flatter shape)
        ellipse = ((xx - cx) / (size * 0.42)) ** 2 + ((yy - cy) / (size * 0.28)) ** 2
        img += np.where(ellipse < 1, 55, 0)

    img += np.random.normal(0, 8, (size, size))  # texture noise
    return np.clip(img, 0, 255).astype(np.uint8)

n_per_class = 300
images, labels = [], []

for _ in range(n_per_class):
    images.append(make_synthetic_image(0))
    labels.append(0)  # cat
for _ in range(n_per_class):
    images.append(make_synthetic_image(1))
    labels.append(1)  # dog

print(f'Total images: {len(images)}')


In [ ]:
# Preview a few sample images
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(images[i], cmap='gray')
    axes[0, i].set_title('Cat (label=0)')
    axes[0, i].axis('off')
for i in range(4):
    axes[1, i].imshow(images[n_per_class + i], cmap='gray')
    axes[1, i].set_title('Dog (label=1)')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()


## 3. Extract Features using HOG (Histogram of Oriented Gradients)

SVMs don't work directly on raw pixels very well for image tasks, so we extract **HOG features** — a classic computer vision technique that captures edge/gradient patterns (shape information), which works well with SVMs for image classification.

In [ ]:
def extract_hog_features(img):
    features, hog_image = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True,
        feature_vector=True
    )
    return features, hog_image

# Visualize HOG on one example
sample_features, sample_hog_img = extract_hog_features(images[0])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(images[0], cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(exposure.rescale_intensity(sample_hog_img, in_range=(0, 10)), cmap='gray')
axes[1].set_title('HOG Features')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'HOG feature vector length: {len(sample_features)}')


In [ ]:
# Extract HOG features for the full dataset
X = np.array([hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)) for img in images])
y = np.array(labels)

print(f'Feature matrix shape: {X.shape}')


## 4. Split into Train / Test Sets and Scale Features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')


## 5. Train the SVM Classifier

We use an RBF kernel, which works well for image features like HOG that have complex, non-linear relationships.

In [ ]:
svm_model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_model.fit(X_train_scaled, y_train)


## 6. Evaluate the Model

In [ ]:
y_pred = svm_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Cat', 'Dog'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()


## 7. Try Different Kernels (Optional Comparison)

It's good practice to compare kernels to justify your final choice.

In [ ]:
for kernel in ['linear', 'rbf', 'poly']:
    model = SVC(kernel=kernel, C=10, gamma='scale', random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f'{kernel} kernel accuracy: {acc:.4f}')


## 8. Predict on a New Image

In [ ]:
new_image = make_synthetic_image(label=1)  # replace with a real loaded+resized+grayscaled image
new_features = hog(new_image, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)).reshape(1, -1)
new_features_scaled = scaler.transform(new_features)

prediction = svm_model.predict(new_features_scaled)[0]
label_name = 'Cat' if prediction == 0 else 'Dog'

plt.imshow(new_image, cmap='gray')
plt.title(f'Predicted: {label_name}')
plt.axis('off')
plt.show()


## Summary

> **Note:** With this synthetic data, accuracy comes out to 100% because the two shapes are artificially very distinct. Real cat/dog photos are far messier (varying poses, backgrounds, lighting), so with the actual Kaggle dataset expect accuracy roughly in the 65-85% range for a classical HOG+SVM approach — that's normal and still a valid result to report. Don't be alarmed if your real-data accuracy is much lower than what you see here.

- We built an SVM classifier to distinguish cats from dogs using **HOG (Histogram of Oriented Gradients)** features extracted from images.
- We scaled features (important for SVM, which is distance/margin-based) and compared different kernels.
- We evaluated the model using accuracy, a classification report, and a confusion matrix.

**Next steps to make this more robust for a real submission:**
- Download the real Kaggle 'Dogs vs. Cats' dataset and use the loading code provided in Section 2
- Try more/fewer HOG parameters (`pixels_per_cell`, `orientations`) and see how accuracy changes
- Try `GridSearchCV` to tune `C` and `gamma` for the SVM
- Compare HOG+SVM against a CNN-based deep learning approach for context on trade-offs (classical ML vs deep learning for images)
